In [ ]:
import logging
from typing import cast

logging.basicConfig(level="DEBUG")
logging.getLogger("dulwich").setLevel(logging.WARNING)
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [ ]:
import torch
from darts.models import LinearRegressionModel

from darts.dataprocessing.transformers import Scaler
from sklearn.preprocessing import StandardScaler

from aare.evaluation.evaluation import evaluate_model
from aare.feature_set import FeatureSet
from aare.features.registry import FEATURES
from aare.feature_identifiers import FeatureIdentifiers
from aare.params import read_params

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()

In [ ]:
import mlflow

mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

# Linear Regression

Even though we know that there are non-linear relationships at play, a simple linear regression model using the last water temperature, the relevant air temperature lag(s), as well as the time, might give a good comparison to other approaches.

In [ ]:
validation_params = params["validation"]
stride = validation_params["stride"]
min_lookback_hours = validation_params["min_lookback_hours"]
forecast_horizon = params["general"]["forecast_horizon"]

In [ ]:
features: FeatureIdentifiers = {
    "targets": ["temp_bern"],
    "future": [
        "tt_bern",
        "tt_bern_log",
        "tt_bern_cube",
        "tt_bern_sqrt",
        "tt_bern_ma3",
        "wind_bern",
        # "tt_bern_ma6",
        # "tt_bern_ma12",
        # "tt_bern_ma24",
        # "tt_bern_ma60",
        # "flow_bern",
        # "ss_bern",
        # "ss_bern_ma3",
        # "ss_bern_ma6",
        # "ss_bern_ma12",
        # "ss_bern_ma24",
        # "ss_bern_ma60",
    ],
}

In [ ]:
ds = FeatureSet(
    targets=FEATURES.get_many(features["targets"]),
    future=FEATURES.get_many(features.get("future")),
    split_params=params["split"],
)

In [ ]:
train = ds.get_train()
train

In [ ]:
val = ds.get_val()
val

In [ ]:
train_target_subs = train[0]
train_fc_subs = train[2]
val_target_subs = val[0]
val_fc_subs = val[2]

In [ ]:
[len(x) for x in train_target_subs]

In [ ]:
train_lens = [len(x) for x in train_target_subs]
val_lens = [len(x) for x in val_target_subs]
data_stats = {
    "train_lens": train_lens,
    "train_len_total": sum(train_lens),
    "train_n_subs": len(train_lens),
    "val_lens": val_lens,
    "val_len_total": sum(val_lens),
    "val_n_subs": len(val_lens),
    "val_split": sum(val_lens) / (sum(val_lens) + sum(train_lens)),
}

In [ ]:
scaler_target = Scaler(StandardScaler(), global_fit=True)
scaler_fc = Scaler(StandardScaler(), global_fit=True)

In [ ]:
scaler_target.fit(train_target_subs)
scaler_fc.fit(train_fc_subs)

In [ ]:
from aare.compat.types import DataTransformers

data_transformers: DataTransformers = {
    "series": scaler_target,
    "future_covariates": scaler_fc,
}

In [ ]:
hparams_model = {
    "lags": [-1, -6, -12, -24],  # use water temp at last hour
    "lags_future_covariates": [0, -1, -6, -12, -24],  # use air temp at last hour (highest corr)
    "likelihood": None,  # "quantile"
    "quantiles": [0.25, 0.5, 0.75],
    "random_state": 42,
    "output_chunk_length": 1,  # only predict 1 hour into the future
    "use_static_covariates": False,  # currently no static covariates
    "multi_models": True,  # 1 model for each output step (doesn't matter if out_chunk_length is 1 anyway)
    # "add_encoders": {"cyclic": {"future": ["hour", "day_of_year"]}},
}

In [ ]:
model = LinearRegressionModel(**hparams_model)

In [ ]:
model.fit(
    series=scaler_target.transform(train_target_subs),
    future_covariates=scaler_fc.transform(train_fc_subs),
)

In [ ]:
metrics, samples = evaluate_model(
    model,
    val_target_subs,
    forecast_horizon,
    stride,
    min_lookback_hours,
    future_cov=val_fc_subs,
    # num_samples=128,
    data_transformers=data_transformers,
    tz=params["general"]["timezone"],
)

In [ ]:
metrics

In [ ]:
_ = samples.plot("LR", with_covariates=["tt_bern"])

In [ ]:
_ = samples.plot("LR", with_covariates=False)

In [ ]:
hparams = (
    {
        "horizon": forecast_horizon,
        "val_stride": stride,
        "split_train": params["split"]["train_split"],
        "split_val": params["split"]["val_split"],
        "split_test": params["split"]["test_split"],
        "features_targets": features["targets"],
        "features_future": features["future"],
    }
    | {"model_" + key: value for key, value in hparams_model.items()}
    | {"data_" + key: value for key, value in data_stats.items()}
)

In [ ]:
mlflow.set_experiment("LR")
with mlflow.start_run() as run:
    mlflow.log_params(hparams)
    mlflow.log_metrics(metrics.to_dict())
    mlflow.log_dict(cast(dict, read_params(ensure_dvc=True)), "params.yaml")

# Store and Load

Just for testing purposes, no need to keep backward compatible

In [ ]:
from aare.storage.model import save_model

save_model("LR", "dev", model, features, data_transformers, run.info)

In [ ]:
from aare.storage.model import load_model

meta, model_loaded, scalers = load_model(name="LR", version="dev")

In [ ]:
meta

In [ ]:
model_loaded

In [ ]:
model_loaded._fit_called